# Notebook: 99 — Manutenção e Consolidação do Corpus Jornalístico (IPSO-H)
**id_rastreabilidade:** RTB_009
**Versão:** v02
**Data de criação:** 28/05/2026
**Última atualização:** 12/06/2026
**Responsável:** Ailton Vendramini

---

## 🎯 Finalidade
Consolidar os CSVs diários do corpus IPSO-H (Tribuna Liberal) num único arquivo canônico, migrando schemas legados para o schema vigente v10.5 **sem alterar os arquivos de origem**.

---

## 📥 Fonte de Dados
**Caminho:** `dados/bd_externos/imprensa/tribuna_liberal/AAAA_MM_DD_tribuna_liberal.csv`
**Base:** Corpus jornalístico IPSO-H — Tribuna Liberal
**Período:** dez/2025 – jun/2026
**Régua:** `regras_de_classificacao_v10.5.md` (core)

---

## ⚙️ Etapa do Pipeline
- [ ] Ingestão (bruto)
- [x] Limpeza (limpo)
- [x] Curadoria (curado)
- [ ] Análise
- [x] Exportação

---

## 📊 Outputs Esperados
| tipo_output | descrição | destino |
|------------|----------|--------|
| utilitario | corpus consolidado canônico | `dados/bd_externos/imprensa/corpus_consolidado_canonico.csv` |

---

## 🧠 Observações
- Notebook **utilitário** (prefixo `99_`, RTB_009 / matriz regra 8) — **não** entra em pipelines analíticos.
- Os CSVs diários são a **fonte de verdade**; o consolidado é **derivado** e idempotente.
- **Schema-alvo v10.5** = 21 colunas, com `aproximacao_variavel` (pos. 14) e `relevancia_estrategica` (pos. 21). **Não** existe `data_classificacao` como campo canônico (ver matriz RTB_019b / P13).
- **4 camadas válidas** (régua v10.5, Seção 3): `IVS`, `GOVERNANCA`, `PRESSAO_SOCIAL`, `CONTEXTO`.
- Os arquivos diários **nunca são sobrescritos**.

## 1. Configuração — caminhos, schema canônico e helpers

In [1]:
import os
import re
import glob
import csv
import pandas as pd

# --- Caminhos absolutos (Windows). No Debian, troque por barras normais. ---
CORPUS_DIR = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\bd_externos\imprensa\tribuna_liberal'
SAIDA      = r'C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\bd_externos\imprensa\corpus_consolidado_canonico.csv'

# Schema canônico v10.5 — 21 colunas, NESTA ordem (matriz RTB_019b).
# Inclui aproximacao_variavel (14) e relevancia_estrategica (21).
# NÃO inclui data_classificacao (não é campo oficial).
SCHEMA_V105 = [
    'id_evento', 'data_publicacao', 'fonte', 'titulo', 'pagina',
    'municipio', 'localidade', 'tipo_camada', 'dimensao_analitica',
    'tipo_relacao_variavel', 'codigo_variavel', 'nivel_criticidade',
    'observacao', 'aproximacao_variavel', 'nota_analista',
    'cod_loteamento', 'nivel_confianca_loteamento', 'papel_no_ciclo',
    'id_caso_pressao', 'entra_ipst', 'relevancia_estrategica',
]

# 4 camadas válidas (régua v10.5, Seção 3).
CAMADAS_VALIDAS = {'IVS', 'GOVERNANCA', 'PRESSAO_SOCIAL', 'CONTEXTO'}

# Vocabulário de relevancia_estrategica (matriz RTB_019b).
RELEVANCIA_VALIDA = {'estrutural', 'alta', 'media', 'baixa'}

# Mapa de aliases: colunas de gerações antigas -> nome canônico v10.5.
# Resolve as gerações que renomearam colunas (mai/2026 em diante).
ALIASES = {
    'data_edicao': 'data_publicacao',
    'data': 'data_publicacao',
    'item': 'id_evento',
    'resumo': 'titulo',
    'titulo_resumido': 'titulo',
    'dimensao_ivs': 'dimensao_analitica',
    'municipio_impactado': 'municipio',
}

# Colunas-fonte cujo conteúdo é RESUMO, não o título original da matéria.
# Linhas que receberem 'titulo' a partir delas são marcadas p/ recuperar do jornal.
FONTES_RESUMO = {'resumo', 'titulo_resumido'}

print(f'Schema v10.5: {len(SCHEMA_V105)} colunas')
print(f'Camadas válidas: {sorted(CAMADAS_VALIDAS)}')
print(f'Aliases de tradução: {len(ALIASES)} mapeamentos')

Schema v10.5: 21 colunas
Camadas válidas: ['CONTEXTO', 'GOVERNANCA', 'IVS', 'PRESSAO_SOCIAL']
Aliases de tradução: 7 mapeamentos


In [2]:
def camada_canonica(v):
    """Normaliza grafia da camada. Retorna 'INVALIDO' se não for uma das 4."""
    v = str(v).strip().upper()
    if v in {'PRESSAO', 'PRESSAO_SOCIAL', 'PRESSÃO SOCIAL', 'PRESSÃO_SOCIAL'}:
        return 'PRESSAO_SOCIAL'
    if v in {'GOVERNANCA', 'GOVERNANÇA'}:
        return 'GOVERNANCA'
    if v == 'CONTEXTO':
        return 'CONTEXTO'
    if v == 'IVS':
        return 'IVS'
    return 'INVALIDO'


def normaliza_data(valor):
    """Converte data para ISO AAAA-MM-DD. Aceita ISO e DD/MM/AAAA.
    Retorna None se não reconhecer."""
    v = str(valor).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}$', v):
        return v
    m = re.match(r'^(\d{2})/(\d{2})/(\d{4})$', v)
    if m:
        d, mes, a = m.groups()
        return f'{a}-{mes}-{d}'
    return None

## 2. Diagnóstico — quantas gerações de schema existem na pasta?

Antes de migrar, este passo **inspeciona** os cabeçalhos reais de todos os diários e os agrupa por "assinatura de schema" (conjunto+ordem de colunas). Mostra quantas gerações distintas existem e quais colunas cada uma tem vs. o v10.5. **Não altera nada.**

In [3]:
arquivos = sorted(glob.glob(os.path.join(CORPUS_DIR, '*.csv')))
# Ignora o próprio consolidado, se estiver na mesma pasta.
arquivos = [a for a in arquivos if 'consolidado' not in os.path.basename(a).lower()]
print(f'{len(arquivos)} CSVs diários encontrados.\n')

# Agrupa arquivos pela assinatura do cabeçalho (tupla ordenada de colunas).
assinaturas = {}
for arq in arquivos:
    with open(arq, encoding='utf-8', newline='') as f:
        cab = tuple(c.strip() for c in next(csv.reader(f)))
    assinaturas.setdefault(cab, []).append(os.path.basename(arq))

print(f'>>> {len(assinaturas)} gerações de schema distintas encontradas.\n')
for i, (cab, lista) in enumerate(
        sorted(assinaturas.items(), key=lambda kv: len(kv[1]), reverse=True), 1):
    faltam = [c for c in SCHEMA_V105 if c not in cab]
    extras = [c for c in cab if c not in SCHEMA_V105]
    print(f'── Geração {i}: {len(lista)} arquivos · {len(cab)} colunas')
    print(f'   Ex.: {lista[0]}  …  {lista[-1]}')
    print(f'   Faltam vs v10.5: {faltam or "(nenhuma)"}')
    print(f'   Extras (irão p/ quarentena _qrt_): {extras or "(nenhuma)"}')
    print()

126 CSVs diários encontrados.

>>> 8 gerações de schema distintas encontradas.

── Geração 1: 109 arquivos · 21 colunas
   Ex.: 2025_12_27_tribuna_liberal.csv  …  2026_06_11_tribuna_liberal.csv
   Faltam vs v10.5: (nenhuma)
   Extras (irão p/ quarentena _qrt_): (nenhuma)

── Geração 2: 4 arquivos · 22 colunas
   Ex.: 2026_05_22_tribuna_liberal.csv  …  2026_05_27_tribuna_liberal.csv
   Faltam vs v10.5: ['data_publicacao', 'titulo', 'municipio', 'localidade', 'nivel_criticidade', 'aproximacao_variavel', 'nivel_confianca_loteamento']
   Extras (irão p/ quarentena _qrt_): ['data_edicao', 'numero_edicao', 'titulo_resumido', 'origem', 'municipio_impactado', 'tipo_abrangencia', 'polaridade_evento', 'tipo_origem_evento']

── Geração 3: 4 arquivos · 21 colunas
   Ex.: 2026_06_04_tribuna_liberal.csv  …  2026_06_09_tribuna_liberal.csv
   Faltam vs v10.5: ['aproximacao_variavel']
   Extras (irão p/ quarentena _qrt_): ['data_classificacao']

── Geração 4: 3 arquivos · 22 colunas
   Ex.: 2026_05_15_

## 3. Migração schema → v10.5 e consolidação

Para cada diário: alinha por **nome** de coluna (linhas bem-formadas nunca deslizam), preenche colunas ausentes com vazio, preserva colunas extras em quarentena `_qrt_`, normaliza a grafia da camada e a data, e marca linhas suspeitas. **Não sobrescreve os originais.**

In [4]:
frames = []
relatorio = []

for arq in arquivos:
    nome = os.path.basename(arq)
    try:
        d = pd.read_csv(arq, sep=',', encoding='utf-8', dtype=str, keep_default_na=False)
    except Exception as e:
        print(f'  ⚠ {nome}: erro de leitura — {e}')
        continue
    d.columns = d.columns.str.strip()

    # Detecta se o título virá de um campo-resumo (gerações antigas).
    titulo_eh_resumo = any(c in FONTES_RESUMO for c in d.columns)

    # Aplica aliases: renomeia colunas antigas p/ o nome canônico, só se a
    # canônica de destino ainda não existir (não sobrescreve coluna presente).
    renomear = {}
    for col_antiga, col_canonica in ALIASES.items():
        if col_antiga in d.columns and col_canonica not in d.columns:
            renomear[col_antiga] = col_canonica
    d = d.rename(columns=renomear)

    extras = [c for c in d.columns if c not in SCHEMA_V105]

    # Reindexa por NOME na ordem canônica; ausentes viram ''.
    out = pd.DataFrame()
    for col in SCHEMA_V105:
        out[col] = d[col] if col in d.columns else ''

    # Preserva extras em quarentena (não descarta dado).
    for c in extras:
        out[f'_qrt_{c}'] = d[c]

    # Normaliza grafia da camada e a data.
    out['tipo_camada'] = out['tipo_camada'].apply(camada_canonica)
    out['data_publicacao'] = out['data_publicacao'].apply(
        lambda v: normaliza_data(v) or v)  # mantém original se não converter

    # Flag: título veio de resumo -> recuperar o título real no jornal depois.
    out['_titulo_pendente'] = 'SIM' if titulo_eh_resumo else ''

    # Marca linhas suspeitas: camada fora das 4 OU data fora de ISO.
    # .astype(str) blinda contra colunas que o pandas inferiu como não-texto.
    cam_ok  = out['tipo_camada'].isin(CAMADAS_VALIDAS)
    data_ok = out['data_publicacao'].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$', na=False)
    out['_linha_suspeita'] = (~(cam_ok & data_ok)).map({True: 'SIM', False: ''})
    out['_arquivo'] = nome

    n_susp = int((out['_linha_suspeita'] == 'SIM').sum())
    relatorio.append((nome, len(out), n_susp, extras))
    frames.append(out)

corpus_canon = pd.concat(frames, ignore_index=True)
print(f'Consolidado em memória: {len(corpus_canon)} eventos de {len(frames)} arquivos.')

Consolidado em memória: 604 eventos de 126 arquivos.


## 4. Relatório de migração e distribuição por camada

In [5]:
print('═'*64)
print('RELATÓRIO DE MIGRAÇÃO')
print('═'*64)
print(f'  Arquivos processados : {len(relatorio)}')
print(f'  Total de eventos     : {len(corpus_canon)}')
n_susp_tot = int((corpus_canon['_linha_suspeita'] == 'SIM').sum())
print(f'  Linhas suspeitas     : {n_susp_tot}  (marcadas, NÃO corrigidas)')
n_tit_pend = int((corpus_canon['_titulo_pendente'] == 'SIM').sum())
print(f'  Títulos pendentes    : {n_tit_pend}  (resumo no lugar do título — recuperar do jornal)')
print()

# Distribuição por camada — só linhas válidas.
val = corpus_canon[corpus_canon['_linha_suspeita'] != 'SIM']
print('Distribuição por camada (linhas válidas):')
tot = len(val)
ordem = ['GOVERNANCA', 'PRESSAO_SOCIAL', 'CONTEXTO', 'IVS']
for cam in ordem:
    n = int((val['tipo_camada'] == cam).sum())
    print(f'  {cam:<16} {n:>4}  ·  {n/tot*100:5.1f}%' if tot else f'  {cam}: 0')
print(f'  {"TOTAL VÁLIDO":<16} {tot:>4}')

════════════════════════════════════════════════════════════════
RELATÓRIO DE MIGRAÇÃO
════════════════════════════════════════════════════════════════
  Arquivos processados : 126
  Total de eventos     : 604
  Linhas suspeitas     : 20  (marcadas, NÃO corrigidas)
  Títulos pendentes    : 42  (resumo no lugar do título — recuperar do jornal)

Distribuição por camada (linhas válidas):
  GOVERNANCA        324  ·   55.5%
  PRESSAO_SOCIAL    146  ·   25.0%
  CONTEXTO          100  ·   17.1%
  IVS                14  ·    2.4%
  TOTAL VÁLIDO      584


In [6]:
# Inspeção dirigida das linhas suspeitas (para correção MANUAL no diário de origem).
susp = corpus_canon[corpus_canon['_linha_suspeita'] == 'SIM']
if len(susp):
    print(f'{len(susp)} linhas suspeitas — corrigir no CSV diário de origem:')
    print('(camada inválida = deslize de coluna; data inválida = vírgula no título)\n')
    cols = ['_arquivo', 'id_evento', 'data_publicacao', 'tipo_camada', 'titulo']
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 40):
        display(susp[cols].astype(str))
else:
    print('Nenhuma linha suspeita. Corpus 100% consistente com o v10.5.')

20 linhas suspeitas — corrigir no CSV diário de origem:
(camada inválida = deslize de coluna; data inválida = vírgula no título)



,_arquivo,id_evento,data_publicacao,tipo_camada,titulo
0,2025_12_27_tribuna_liberal.csv,TL_20251227_001,2025-12-27,INVALIDO,"""Família relata ameaças"
4,2025_12_28_tribuna_liberal.csv,TL_20251228_001,2025-12-28,INVALIDO,"""Sumaré e Hortolândia terão 50"
10,2025_12_29_tribuna_liberal.csv,TL_20251229_003,2025-12-29,INVALIDO,"""Estrada Pedrina Guilherme terá 2"
58,2026_01_10_tribuna_liberal.csv,TL_20260110_003,2026-01-10,INVALIDO,"""Motiva AutoBAn transfere R$ 4.585.971"
83,2026_01_17_tribuna_liberal.csv,TL_20260117_001,2026-01-17,INVALIDO,Chuvas intensas de 93
110,2026_01_24_tribuna_liberal.csv,TL_20260124_001,2026-01-24,INVALIDO,Repasse FEAS aumenta para R$ 1.479 m...
139,2026_01_31_tribuna_liberal.csv,TL_20260131_02,2026-01-31,INVALIDO,"""Monte Mor decreta emergencia apos 125"
140,2026_01_31_tribuna_liberal.csv,TL_20260131_03,2026-01-31,INVALIDO,"""Regiao recebe R$ 198"
144,2026_02_01_tribuna_liberal.csv,E01,2026-02-01,INVALIDO,"""Hortolândia divulga pré-selecionado..."
150,2026_02_03_tribuna_liberal.csv,E05,2026-02-03,INVALIDO,"""Ações em Hortolândia estimulam cons..."


## 5. Validação e gravação

Só grava se as checagens passarem. Escrita com `QUOTE_ALL` (protege vírgulas internas — o erro do pipeline antigo).

In [7]:
problemas = []

# (1) Colunas canônicas presentes e na ordem certa (ignorando _qrt_ / _ técnicos).
cols_canon = [c for c in corpus_canon.columns if not c.startswith('_')]
if cols_canon != SCHEMA_V105:
    problemas.append('Ordem/conjunto das 21 colunas canônicas diverge do v10.5.')

# (2) relevancia_estrategica dentro do vocabulário (ignora vazio).
rel_fora = set(corpus_canon['relevancia_estrategica'].astype(str).unique()) - RELEVANCIA_VALIDA - {''}
if rel_fora:
    problemas.append(f'relevancia_estrategica fora do vocabulário: {rel_fora}')

if problemas:
    print('=== PROBLEMAS (resolver antes de gravar) ===')
    for p in problemas:
        print(' -', p)
else:
    print('Validação OK — pronto para gravar.')

Validação OK — pronto para gravar.


In [8]:
if problemas:
    print('NÃO gravado: resolva os problemas acima primeiro.')
else:
    corpus_canon.to_csv(SAIDA, index=False, quoting=csv.QUOTE_ALL, encoding='utf-8')
    print(f'✓ Consolidado canônico salvo:\n  {SAIDA}')
    print(f'  {corpus_canon.shape[0]} linhas x {corpus_canon.shape[1]} colunas')
    print('  (os CSVs diários NÃO foram alterados)')

    # Verificação de ida e volta.
    rele = pd.read_csv(SAIDA, dtype=str, keep_default_na=False)
    assert rele.shape[0] == corpus_canon.shape[0], 'Contagem relida difere!'
    print(f'  Ida e volta OK — releitura: {rele.shape[0]} linhas.')

✓ Consolidado canônico salvo:
  C:\Users\ailto\Atlas-Social-de-Hortolandia\dados\bd_externos\imprensa\corpus_consolidado_canonico.csv
  604 linhas x 36 colunas
  (os CSVs diários NÃO foram alterados)
  Ida e volta OK — releitura: 604 linhas.
